# Frozen ResNet18 + BiGRU With Speaker-Word Balanced Sampling

Третий контролируемый эксперимент. Архитектура и frozen ImageNet features остаются теми же; меняется только sampling train-примеров.

Текущий train перекошен: `spk04` даёт большую часть примеров, а слово `есть` встречается чаще остальных. Поэтому каждому непустому сочетанию `(speaker, word)` назначается одинаковая суммарная вероятность попадания в train epoch.

Это дешёвый тест гипотезы о дисбалансе. Feature caches уже лежат на Google Drive, повторно извлекать ResNet-признаки и распаковывать видео не нужно. Test split не загружается.

## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Загрузить frozen ResNet feature caches

In [ ]:
from pathlib import Path
from collections import Counter
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


SEED = 42
set_reproducible_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

feature_root = Path(
    '/content/drive/MyDrive/lipreading_sem4/controlled_frozen_resnet18_bigru'
)
train_cache_path = feature_root / 'dataset_v1_train_resnet18_imagenet_features.pt'
val_cache_path = feature_root / 'dataset_v1_val_resnet18_imagenet_features.pt'

if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Frozen ResNet feature caches were not found. '
        'Run lipreading_frozen_resnet18_bigru_colab.ipynb first.'
    )

train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)

word_label_pairs = set(zip(train_cache['words'], train_cache['labels'].tolist()))
word_to_idx = {word: int(label) for word, label in word_label_pairs}
idx_to_word = {index: word for word, index in word_to_idx.items()}
vocab = [idx_to_word[index] for index in sorted(idx_to_word)]

checkpoint_dir = Path(
    '/content/drive/MyDrive/lipreading_sem4/controlled_frozen_resnet18_bigru_balanced'
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print('device:', device)
print('train features:', train_cache['features'].shape)
print('validation features:', val_cache['features'].shape)
print('vocab:', vocab)
print('results directory:', checkpoint_dir)
print('TEST SPLIT IS NOT LOADED.')

## 3. Посмотреть исходный дисбаланс

In [ ]:
train_metadata = pd.DataFrame({
    'clip_path': train_cache['clip_paths'],
    'word': train_cache['words'],
    'speaker_id': train_cache['speakers'],
    'label': train_cache['labels'].tolist(),
})
val_metadata = pd.DataFrame({
    'clip_path': val_cache['clip_paths'],
    'word': val_cache['words'],
    'speaker_id': val_cache['speakers'],
    'label': val_cache['labels'].tolist(),
})

speaker_counts = train_metadata['speaker_id'].value_counts().rename('clips')
word_counts = train_metadata['word'].value_counts().rename('clips')
speaker_word_counts = (
    train_metadata.groupby(['speaker_id', 'word'])
    .size()
    .rename('clips')
    .reset_index()
    .sort_values(['speaker_id', 'word'])
)

print('train clips by speaker:')
display(speaker_counts.to_frame())
print('train clips by word:')
display(word_counts.to_frame())
print('train clips by speaker + word:')
display(speaker_word_counts)

majority_word = val_metadata['word'].value_counts().index[0]
majority_baseline_acc = val_metadata['word'].value_counts().iloc[0] / len(val_metadata)
print('validation majority baseline:', majority_word, round(majority_baseline_acc, 4))

## 4. Speaker-word balanced sampler и DataLoader

In [ ]:
class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = cache['labels'].long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


group_keys = list(zip(train_metadata['speaker_id'], train_metadata['word']))
group_counts = Counter(group_keys)
sample_weights = torch.tensor(
    [1.0 / group_counts[key] for key in group_keys],
    dtype=torch.double,
)
sampler_generator = torch.Generator().manual_seed(SEED)
balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=sampler_generator,
)

train_dataset = FeatureSequenceDataset(train_cache)
val_dataset = FeatureSequenceDataset(val_cache)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=balanced_sampler,
    num_workers=0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

expected_groups = len(group_counts)
print('non-empty speaker-word groups:', expected_groups)
print('sample weight min/max:', sample_weights.min().item(), sample_weights.max().item())
print('Each non-empty speaker-word group now has equal total sampling probability.')

feature_batch, label_batch = next(iter(train_loader))
print('feature batch:', feature_batch.shape)
print('label batch:', label_batch.shape)

## 5. Та же модель BiGRU

In [ ]:
class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


set_reproducible_seed(SEED)
model = BiGRUClassifier(num_classes=len(vocab)).to(device)
print('trainable BiGRU/classifier parameters:', count_parameters(model))

with torch.no_grad():
    logits = model(feature_batch.to(device))
print('logits:', logits.shape)

## 6. Метрики и training loop

In [ ]:
def confusion_matrix_array(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1
    return matrix


def classification_metrics(preds, labels, num_classes):
    matrix = confusion_matrix_array(preds, labels, num_classes)
    total = int(matrix.sum())
    accuracy = float(np.trace(matrix) / total) if total else 0.0
    class_rows = []

    for class_id in range(num_classes):
        true_positive = int(matrix[class_id, class_id])
        support = int(matrix[class_id, :].sum())
        predicted_count = int(matrix[:, class_id].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'support': support,
            'predicted_count': predicted_count,
            'correct': true_positive,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active_rows = [row for row in class_rows if row['support'] > 0]
    macro_f1 = float(np.mean([row['f1'] for row in active_rows])) if active_rows else 0.0
    balanced_accuracy = float(np.mean([row['recall'] for row in active_rows])) if active_rows else 0.0
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_accuracy,
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        correct += (outputs.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate_detailed(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item() * batch_x.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())

    metrics = classification_metrics(preds, labels, num_classes)
    metrics['loss'] = total_loss / total
    metrics['preds'] = preds
    metrics['labels'] = labels
    return metrics


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def fit_controlled_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=30,
    patience=7,
):
    best_checkpoint_path = checkpoint_dir / 'frozen_resnet18_bigru_speaker_word_balanced_best.pt'
    history_path = checkpoint_dir / 'frozen_resnet18_bigru_speaker_word_balanced_history.csv'
    history_rows = []
    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    total_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate_detailed(model, val_loader, criterion, device, len(vocab))
        epoch_seconds = time.perf_counter() - epoch_start

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_balanced_acc': val_metrics['balanced_accuracy'],
            'epoch_seconds': epoch_seconds,
        }
        history_rows.append(row)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)

        print(
            f"epoch {epoch}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | {epoch_seconds:.1f}s"
        )

        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )
        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            epochs_without_improvement = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'best_val_macro_f1': best_macro_f1,
                'best_val_acc': val_metrics['accuracy'],
                'best_val_balanced_accuracy': val_metrics['balanced_accuracy'],
                'best_val_loss': best_val_loss,
                'trainable_parameters': count_parameters(model),
                'history': history_rows,
            }, best_checkpoint_path)
            print('saved best by validation macro-F1:', best_checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'early stop: no macro-F1 improvement for {patience} epochs')
            break

    total_seconds = time.perf_counter() - total_start
    return pd.DataFrame(history_rows), best_checkpoint_path, history_path, total_seconds

## 7. Обучение

In [ ]:
set_reproducible_seed(SEED)
model = BiGRUClassifier(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

history_df, best_checkpoint_path, history_path, total_training_seconds = fit_controlled_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=30,
    patience=7,
)

print('epochs completed:', len(history_df))
print('training minutes:', round(total_training_seconds / 60, 2))
print('history:', history_path)

## 8. Оценка на validation speaker spk06

In [ ]:
checkpoint = load_torch_file(best_checkpoint_path, device=device)
best_model = BiGRUClassifier(num_classes=len(vocab)).to(device)
best_model.load_state_dict(checkpoint['model_state_dict'])
val_metrics = evaluate_detailed(best_model, val_loader, criterion, device, len(vocab))

print('best epoch:', checkpoint['epoch'])
print('validation accuracy:', round(val_metrics['accuracy'], 4))
print('validation macro-F1:', round(val_metrics['macro_f1'], 4))
print('validation balanced accuracy:', round(val_metrics['balanced_accuracy'], 4))
print('majority baseline accuracy:', round(majority_baseline_acc, 4))

class_report_rows = []
for row in val_metrics['class_rows']:
    class_report_rows.append({
        'word': idx_to_word[row['class_id']],
        'support': row['support'],
        'predicted_count': row['predicted_count'],
        'correct': row['correct'],
        'precision': row['precision'],
        'recall': row['recall'],
        'f1': row['f1'],
    })
class_report_df = pd.DataFrame(class_report_rows)
display(class_report_df)

prediction_counts = Counter(idx_to_word[index] for index in val_metrics['preds'])
print('predicted word distribution:')
print(pd.Series(prediction_counts).sort_values(ascending=False))

validation_predictions_df = val_metadata.copy()
validation_predictions_df['pred_id'] = val_metrics['preds']
validation_predictions_df['pred_word'] = [idx_to_word[index] for index in val_metrics['preds']]
validation_predictions_df['correct'] = (
    validation_predictions_df['label'] == validation_predictions_df['pred_id']
)

class_report_path = checkpoint_dir / 'balanced_class_report.csv'
predictions_path = checkpoint_dir / 'balanced_val_predictions.csv'
class_report_df.to_csv(class_report_path, index=False)
validation_predictions_df.to_csv(predictions_path, index=False)
print('saved class report:', class_report_path)
print('saved predictions:', predictions_path)

## 9. Графики и confusion matrix

In [ ]:
curves_path = checkpoint_dir / 'balanced_training_curves.png'
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train accuracy')
axes[1].plot(history_df['epoch'], history_df['val_acc'], label='validation accuracy')
axes[1].axhline(majority_baseline_acc, color='gray', linestyle='--', label='majority baseline')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['val_macro_f1'], label='macro-F1')
axes[2].plot(history_df['epoch'], history_df['val_balanced_acc'], label='balanced accuracy')
axes[2].set_title('Balanced validation metrics')
axes[2].set_xlabel('epoch')
axes[2].legend()
fig.tight_layout()
fig.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()

matrix = val_metrics['confusion_matrix']
confusion_path = checkpoint_dir / 'balanced_confusion_matrix.png'
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix, cmap='Blues')
ax.set_title('Frozen ResNet18 + BiGRU + speaker-word balanced sampler')
ax.set_xlabel('predicted word')
ax.set_ylabel('true word')
ax.set_xticks(range(len(vocab)), vocab, rotation=45, ha='right')
ax.set_yticks(range(len(vocab)), vocab)
fig.colorbar(image, ax=ax)
for true_index in range(matrix.shape[0]):
    for pred_index in range(matrix.shape[1]):
        if matrix[true_index, pred_index]:
            ax.text(pred_index, true_index, str(matrix[true_index, pred_index]), ha='center', va='center')
fig.tight_layout()
fig.savefig(confusion_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved curves:', curves_path)
print('saved confusion matrix:', confusion_path)

## 10. Сравнение трёх экспериментов

In [ ]:
predicted_class_count = len(set(val_metrics['preds']))
balanced_summary = pd.DataFrame([{
    'model': 'Frozen ImageNet ResNet18 features + BiGRU + speaker-word sampler',
    'split': 'speaker-based',
    'best_epoch': checkpoint['epoch'],
    'validation_accuracy': val_metrics['accuracy'],
    'validation_macro_f1': val_metrics['macro_f1'],
    'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
    'majority_baseline_accuracy': majority_baseline_acc,
    'predicted_classes': predicted_class_count,
    'trainable_parameters': checkpoint['trainable_parameters'],
    'training_minutes': total_training_seconds / 60,
    'test_used': False,
}])
summary_path = checkpoint_dir / 'balanced_summary.csv'
balanced_summary.to_csv(summary_path, index=False)
display(balanced_summary)

comparison_paths = [
    Path('/content/drive/MyDrive/lipreading_sem4/controlled_framecnn_bigru/framecnn_bigru_gray_controlled_summary.csv'),
    Path('/content/drive/MyDrive/lipreading_sem4/controlled_frozen_resnet18_bigru/frozen_resnet18_bigru_summary.csv'),
]
comparison_frames = [pd.read_csv(path) for path in comparison_paths if path.exists()]
comparison_frames.append(balanced_summary)
comparison = pd.concat(comparison_frames, ignore_index=True)
display(comparison[
    ['model', 'validation_accuracy', 'validation_macro_f1',
     'validation_balanced_accuracy', 'predicted_classes', 'test_used']
])

print('saved summary:', summary_path)
print('TEST SPLIT WAS NOT USED.')

## Что прислать

Скачай выполненный `.ipynb` и пришли его сюда. Результаты сохраняются в:

```text
/content/drive/MyDrive/lipreading_sem4/controlled_frozen_resnet18_bigru_balanced
```